# Exploring the OpenProblemAtlas

This notebook demonstrates how to load, explore, and analyze the atlas data.

In [ ]:
import yaml
from pathlib import Path
from collections import Counter

ROOT = Path('..') 
PROBLEMS_DIR = ROOT / 'data' / 'problems'

In [ ]:
# Load all problems
problems = []
for f in sorted(PROBLEMS_DIR.rglob('*.yaml')):
    with open(f) as fp:
        data = yaml.safe_load(fp)
        if data:
            problems.append(data)

print(f'Loaded {len(problems)} problems')

In [ ]:
# Domain distribution
domain_counts = Counter(p['domain'] for p in problems)
print('Problems by domain:')
for domain, count in domain_counts.most_common():
    print(f'  {domain}: {count}')

In [ ]:
# Status distribution
status_counts = Counter(p['status']['label'] for p in problems)
print('Problems by status:')
for status, count in status_counts.most_common():
    print(f'  {status}: {count}')

In [ ]:
# Subdomain distribution
subdomain_counts = Counter()
for p in problems:
    for sd in p.get('subdomains', []):
        subdomain_counts[sd] += 1

print('Top 15 subdomains:')
for sd, count in subdomain_counts.most_common(15):
    print(f'  {sd}: {count}')

In [ ]:
# Find underexplored problems (high impact, high underexplored score)
underexplored = [
    p for p in problems
    if p.get('scores', {}).get('underexplored', 0) > 0.6
    and p.get('scores', {}).get('impact', 0) > 0.5
    and p['status']['label'] == 'open'
]

print(f'\nUnderexplored problems ({len(underexplored)}):')
for p in underexplored:
    scores = p.get('scores', {})
    print(f"  {p['title']}")
    print(f"    impact={scores.get('impact', '?')}, underexplored={scores.get('underexplored', '?')}")

In [ ]:
# Find solver-ready problems
solver_ready = [
    p for p in problems
    if p.get('ai', {}).get('solver_ready', False)
    and p['status']['label'] == 'open'
]

print(f'Solver-ready problems ({len(solver_ready)}):')
for p in solver_ready:
    tools = p.get('ai', {}).get('recommended_tools', [])
    print(f"  {p['title']} — tools: {', '.join(tools)}")

In [ ]:
# Find formalization-wanted problems
formalization_wanted = [
    p for p in problems
    if p.get('formalization', {}).get('wanted', False)
    and p['status']['label'] == 'open'
]

print(f'Formalization wanted ({len(formalization_wanted)}):')
for p in formalization_wanted:
    print(f"  {p['title']}")

In [ ]:
# Score distributions
score_names = ['impact', 'underexplored', 'toolability', 'formality', 'ai_fit']
print('Score statistics (mean / min / max):')
for name in score_names:
    values = [p.get('scores', {}).get(name, 0) for p in problems if p.get('scores', {}).get(name) is not None]
    if values:
        print(f'  {name:15s}: mean={sum(values)/len(values):.2f}  min={min(values):.2f}  max={max(values):.2f}')